In [1]:

# Basic calculations script
import numpy as np
import matplotlib.pyplot as plt

# Vehicle parameters (estimate for Formula Student car)
mass = 300  # kg (with driver)
wheelbase = 1.53  # meters
cg_height = 0.3  # meters
weight_dist_front = 0.49  # 49% front weight distribution
total_weight = mass * 9.81  # N
Fx_front = 1828.2 #N Maximum longitudinal force at the tire contact patch (from tire data)
front_tire_friction_coefficient = 1.63
gravity = 9.81

# Calculate static weight on each axle
weight_front = mass * gravity * weight_dist_front
weight_rear = mass * gravity * (1 - weight_dist_front)

print(f"Static front weight: {weight_front:.1f} N")
print(f"Static rear weight: {weight_rear:.1f} N")

# Calculate longitudinal weight transfer during braking
def calculate_weight_transfer(mass, deceleration_g, cg_height, wheelbase):
    """
    Calculate longitudinal weight transfer during braking
    """
    weight_transfer = (total_weight * deceleration_g * cg_height) / wheelbase
    return weight_transfer

# Test with 1.3g braking (Max Fz tested in the tire data)
decels = np.linspace(0.9, 1.6, 5)
transfers = [calculate_weight_transfer(mass, d, cg_height, wheelbase) for d in decels]
print("\nDeceleration (g)  |  Weight Transfer (N)  |  % of Total Weight")
print("-" * 60)
for d, t in zip(decels, transfers):
    print(f"      {d:5.2f}       |       {t:6.1f}       |       {t / total_weight * 100:5.1f}% ")
"""
# Plotting weight transfer vs deceleration
plt.plot(decels, transfers)
plt.xlabel('Deceleration (g)')
plt.ylabel('Weight Transfer (N)')
plt.title('Longitudinal Weight Transfer During Braking')
plt.grid(True)
plt.show()
"""
def front_axle_load(weight_front, weight_transfer):
    """
    Calculate front axle load during braking
    """
    return weight_front + weight_transfer

def rear_axle_load(weight_rear, weight_transfer):
    """
    Calculate rear axle load during braking
    """
    return weight_rear - weight_transfer

print("\nDeceleration (g)  |  Front Axle Load (N)  |  Rear Axle Load (N)")
print("-" * 135)
for d, t in zip(decels, transfers):
    front_load = front_axle_load(weight_front, t)
    rear_load = rear_axle_load(weight_rear, t)
    print(f"      {d:5.2f}       |       {front_load:6.1f} N       |       {rear_load:6.1f} N")

#Precise deceleration for a target vertical load (Fz) using interpolation
decel_g = np.array([1.3, 1.4])
Fz = np.array([2192.2, 2250])
Fz_target = 2224
decel_target = np.interp(Fz_target, Fz, decel_g)

print(f"\nDeceleration for Fz = {Fz_target} N : Deceleration = {decel_target:.2f} g")
print(f"This is just to know the precise deceleration for maximum vertical load tested in the tire data.")

#Max deceleration for the target vertical load
max_decel = front_tire_friction_coefficient * gravity
print(f"\nMax deceleration based on tire friction coefficient: {max_decel:.2f} m/s^2")

#Brake Balance Calculation
X1 = front_tire_friction_coefficient * (weight_front + ((mass * cg_height)/wheelbase) * max_decel)
X2 = front_tire_friction_coefficient * (weight_rear - ((mass * cg_height)/wheelbase) * max_decel)
print(f"\nBrake Balance Calculation:")
print(f"Front axle braking force: {X1:.1f} N")
print(f"Rear axle braking force: {X2:.1f} N")
print(f"Brake balance: {X1/(X1+X2)*100:.1f}% front")

#Pratical Brake Balance
weight_distribution = weight_front / (weight_front + weight_rear)
pratical_BB = ((total_weight * (front_tire_friction_coefficient * (cg_height / wheelbase) + weight_distribution)) / (total_weight)) * 100
print(f"Pratical Brake Balance: {pratical_BB:.1f}%") #Just a different way of calculation the Brake Balance, either way is right and is also a good practice to make all calculations are getting the same result



Static front weight: 1442.1 N
Static rear weight: 1500.9 N

Deceleration (g)  |  Weight Transfer (N)  |  % of Total Weight
------------------------------------------------------------
       0.90       |        519.4       |        17.6% 
       1.07       |        620.3       |        21.1% 
       1.25       |        721.3       |        24.5% 
       1.43       |        822.3       |        27.9% 
       1.60       |        923.3       |        31.4% 

Deceleration (g)  |  Front Axle Load (N)  |  Rear Axle Load (N)
---------------------------------------------------------------------------------------------------------------------------------------
       0.90       |       1961.4 N       |        981.6 N
       1.07       |       2062.4 N       |        880.6 N
       1.25       |       2163.4 N       |        779.6 N
       1.43       |       2264.4 N       |        678.6 N
       1.60       |       2365.4 N       |        577.6 N

Deceleration for Fz = 2224 N : Deceleration = 1.3